This script follows https://github.com/cosmodesi/desi-clustering/blob/main/local_png/check_mocks.ipynb. 

In [1]:
import os
import re
import numpy as np
from pathlib import Path

import pandas as pd
from itertools import product

from clustering_statistics.tools import get_stats_fn, base_stats_dir

In [2]:
def _get_available_mock_ids(base_dir):
    """
    Inventory the mock directories available in the project.

    Returns:
        mock_ids_expected: list of all expected mock IDs (0-999)
        mock_ids_present: list of actually present mock IDs
        mock_ids_missing: list of missing mock IDs
    """
    mock_min, mock_max = 0, 2000
    mock_ids_present = sorted(int(m.group(1)) for p in base_dir.iterdir() if p.is_dir() and (m := re.match(r'^mock(\d+)$', p.name)))
    if not mock_ids_present: raise RuntimeError(f'No mockN directory found in {base_dir}')
    mock_ids_expected = list(range(mock_min, mock_max + 1))
    mock_ids_missing = sorted(set(mock_ids_expected) - set(mock_ids_present))
    print(f'Discovered {len(mock_ids_present)} mock directories over 1000 expected (0-{mock_max})')
    return mock_ids_expected, mock_ids_present, mock_ids_missing
    
def _discover_available_by_stats(base_dir, stats_kind, data_type):
    """
    Discover available tracers, redshift ranges, regions, and weights by scanning existing files.

    Args:
        base_dir: Path to the project directory containing mockN subdirectories
        stats_kind: single stats kind string
        data_type: whether to search data or mocks

    Returns:
        tracers_available: sorted list of tracers
        zranges_by_tracer: dict mapping tracer -> sorted list of z-range tuples
        regions_by_tracer: dict mapping tracer -> sorted list of regions
        weights_by_tracer: dict mapping tracer -> sorted list of weights
    """
    tracers_available, zranges_by_tracer, regions_by_tracer, weights_by_tracer = set(), {}, {}, {}
    escaped_kind = re.escape(stats_kind)
    pattern = re.compile(rf'^{escaped_kind}_(?P<tracer>.+?)_z(?P<zmin>\d+(?:\.\d+)?)-(?P<zmax>\d+(?:\.\d+)?)_(?P<region>[^_]+)_weight-(?P<weight>.+)\.h5$')
    base_dir_glob = base_dir.glob('mock*') if data_type == 'mock' else [base_dir]
    for mock_dir in base_dir_glob:
        if not mock_dir.is_dir(): continue
        for fn in mock_dir.glob(f'{stats_kind}_*.h5'):
            match = pattern.match(fn.name)
            if not match: continue
            tracer = match.group('tracer')
            raw_weight = match.group('weight')
            weight = raw_weight.removesuffix('_auw').removesuffix('_thetacut')
            zrange = (float(match.group('zmin')), float(match.group('zmax')))
            region = match.group('region')
            tracers_available.add(tracer)
            zranges_by_tracer.setdefault(tracer, set()).add(zrange)
            regions_by_tracer.setdefault(tracer, set()).add(region)
            weights_by_tracer.setdefault(tracer, set()).add(weight)
    tracers_available = sorted(tracers_available)
    zranges_by_tracer = {t: sorted(zranges_by_tracer.get(t, set())) for t in tracers_available}
    regions_by_tracer = {t: sorted(regions_by_tracer.get(t, set())) for t in tracers_available}
    weights_by_tracer = {t: sorted(weights_by_tracer.get(t, set())) for t in tracers_available}
    return tracers_available, zranges_by_tracer, regions_by_tracer, weights_by_tracer

def get_inventory(stats_dir, project, version, stats_kind='mesh2_spectrum_poles'):
    """
    Get an inventory of expected stats files across mocks, tracers, z-ranges, regions, and weights.

    Args:
        stats_dir: Path to the base stats directory
        project: project name
        version: version of the mocks
        stats_kind: str or list of str indicating stats kinds to scan

    Returns:
        pd.DataFrame with columns: kind, tracer, weight, imock, zrange, region, exists, path
    """
    base_dir = stats_dir / project / version
    data_type = 'data' if 'data' in version else 'mock'
    if isinstance(stats_kind, str): stats_kind = [stats_kind]
    print(f'Directory: {base_dir} -- scanning for stats files of kinds {stats_kind}...')
    # 1) Inventory of available directories
    if data_type == 'mock':
        mock_ids_expected, mock_ids_present, mock_ids_missing = _get_available_mock_ids(base_dir)
    all_rows = []

    for kind in stats_kind:
        # 2) Discover available tracers, z-ranges, regions, weights
        tracers_available, zranges_by_tracer, regions_by_tracer, weights_by_tracer = _discover_available_by_stats(base_dir=base_dir, stats_kind=kind, data_type=data_type)
        
        # 3) Scan existing files once
        all_files = {f.relative_to(stats_dir).as_posix() for f in base_dir.rglob(f'{kind}_*.h5')}
        
        # 4) Generate all expected combinations
        for tracer in tracers_available:
            tracer_zranges = zranges_by_tracer.get(tracer, [])
            tracer_regions = regions_by_tracer.get(tracer, [])
            tracer_weights = weights_by_tracer.get(tracer, [])
            if not tracer_zranges or not tracer_regions or not tracer_weights: continue
            if data_type == 'mock':
                for imock, zrange, region, weight in product(mock_ids_expected, tracer_zranges, tracer_regions, tracer_weights):
                    
                    fn = get_stats_fn(stats_dir=stats_dir, project=project+'/'+version, kind=kind, tracer=tracer, region=region, zrange=zrange,
                                      weight=weight, basis='sugiyama-diagonal', imock=imock, auw=False, cut=False)
                    relpath = fn.relative_to(stats_dir).as_posix()
                    
                    all_rows.append({'version': version, 'project': project, 'tracer': tracer, 'kind': kind, 'weight': weight, 
                                     'imock': imock,'zrange': f'{zrange[0]}-{zrange[1]}', 'region': region, 'data type': data_type,
                                     'exists': relpath in all_files, 'directory': str(base_dir), 'path': str(fn)})
            else:
                for zrange, region, weight in product(tracer_zranges, tracer_regions, tracer_weights):
                    
                    fn = get_stats_fn(stats_dir=stats_dir, project=project+'/'+version, kind=kind, tracer=tracer, region=region, zrange=zrange,
                                      weight=weight, basis='sugiyama-diagonal', auw=False, cut=False)
                    
                    relpath = fn.relative_to(stats_dir).as_posix()
                    all_rows.append({'version': version, 'project': project, 'tracer': tracer, 'kind': kind, 'weight': weight, 
                                     'imock': 'NA','zrange': f'{zrange[0]}-{zrange[1]}', 'region': region, 'data type': data_type,
                                     'exists': relpath in all_files, 'directory': str(base_dir), 'path': str(fn)})
                

    return pd.DataFrame(all_rows)

In [3]:
# !ls /global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/local_png/base/holi-v3-altmtl/
# !tree -d -L 3 $base_stats_dir | grep -vE '(complete)'

In [4]:
%%time
stats_dir = base_stats_dir
projects = ['full_shape/base', 'local_png/base', 'full_shape/data_splits']
versions = ['holi-v3-altmtl', 'holi-bgs-altmtl', 'glam-uchuu-v2-altmtl', 'glam-uchuu-bgs-altmtl', 
            'abacus-2ndgen-dr2-altmtl', 'abacus-hf-dr2-v2-altmtl', 'uchuu-hf-altmtl', 'uchuu-hf-reference',
            'blinded_data/data-dr2-v2','desi-data/loa-v1/v2/fNL/blinded']
stats_kind = ['mesh2_spectrum_poles','mesh3_spectrum_sugiyama-diagonal_poles','particle2_correlation_smu']

list_of_tables = []

for project in projects:
    for version in versions:
        base_dir = stats_dir / project / version
        # skip if base dir does not exist
        if not base_dir.is_dir(): continue
        inventory = get_inventory(stats_dir=stats_dir, project=project, version=version, stats_kind=stats_kind)
        available = inventory[inventory['exists']]
        summary = (inventory.groupby(['project', 'version', 'tracer', 'kind', 'weight', 'zrange', 'region', 'data type', 'directory'], as_index=False).agg(available=('exists', 'sum')))
        summary = summary[~summary['tracer'].str.contains('jack')] # remove jackknife measurements from summary
        summary = summary[summary['available'] > 0]
        list_of_tables.append(summary)
        print()

Directory: /global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/full_shape/base/holi-v3-altmtl -- scanning for stats files of kinds ['mesh2_spectrum_poles', 'mesh3_spectrum_sugiyama-diagonal_poles', 'particle2_correlation_smu']...
Discovered 859 mock directories over 1000 expected (0-2000)

Directory: /global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/full_shape/base/holi-bgs-altmtl -- scanning for stats files of kinds ['mesh2_spectrum_poles', 'mesh3_spectrum_sugiyama-diagonal_poles', 'particle2_correlation_smu']...
Discovered 1000 mock directories over 1000 expected (0-2000)

Directory: /global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/full_shape/base/glam-uchuu-v2-altmtl -- scanning for stats files of kinds ['mesh2_spectrum_poles', 'mesh3_spectrum_sugiyama-diagonal_poles', 'particle2_correlation_smu']...
Discovered 983 mock directories over 1000 expected (0-2000)

Directory: /global/cfs/cdirs/desi/science/cai/desi-cl

In [6]:
columns = ['project', 'version', 'tracer', 'kind', 'weight', 'zrange', 'region', 'available', 'data type', 'directory']
full_table = pd.concat(list_of_tables, ignore_index=True)
full_table = full_table[full_table['available'] != 0] # remove unavailable measurements
full_table = full_table[columns] # reorder columns
full_table.to_csv("../helper_scripts/available_measurements.csv", index=False)

In [7]:
full_table

,project,version,tracer,kind,weight,zrange,region,available,data type,directory
0,full_shape/base,holi-v3-altmtl,ELG_LOPnotqso,mesh2_spectrum_poles,default-FKP,0.8-1.1,GCcomb,859,mock,/global/cfs/cdirs/desi/science/cai/desi-cluste...
1,full_shape/base,holi-v3-altmtl,ELG_LOPnotqso,mesh2_spectrum_poles,default-FKP,0.8-1.1,NGC,859,mock,/global/cfs/cdirs/desi/science/cai/desi-cluste...
2,full_shape/base,holi-v3-altmtl,ELG_LOPnotqso,mesh2_spectrum_poles,default-FKP,0.8-1.1,SGC,859,mock,/global/cfs/cdirs/desi/science/cai/desi-cluste...
3,full_shape/base,holi-v3-altmtl,ELG_LOPnotqso,mesh2_spectrum_poles,default-FKP,1.1-1.6,GCcomb,859,mock,/global/cfs/cdirs/desi/science/cai/desi-cluste...
4,full_shape/base,holi-v3-altmtl,ELG_LOPnotqso,mesh2_spectrum_poles,default-FKP,1.1-1.6,NGC,859,mock,/global/cfs/cdirs/desi/science/cai/desi-cluste...
...,...,...,...,...,...,...,...,...,...,...
928,full_shape/data_splits,blinded_data/data-dr2-v2,QSO,mesh3_spectrum_sugiyama-diagonal_poles,default-FKP,0.8-2.1,NGCnoN,1,data,/global/cfs/cdirs/desi/science/cai/desi-cluste...
929,full_shape/data_splits,blinded_data/data-dr2-v2,QSO,mesh3_spectrum_sugiyama-diagonal_poles,default-FKP,0.8-2.1,NS,1,data,/global/cfs/cdirs/desi/science/cai/desi-cluste...
930,full_shape/data_splits,blinded_data/data-dr2-v2,QSO,mesh3_spectrum_sugiyama-diagonal_poles,default-FKP,0.8-2.1,S,1,data,/global/cfs/cdirs/desi/science/cai/desi-cluste...
931,full_shape/data_splits,blinded_data/data-dr2-v2,QSO,mesh3_spectrum_sugiyama-diagonal_poles,default-FKP,0.8-2.1,SGC,1,data,/global/cfs/cdirs/desi/science/cai/desi-cluste...
